<a href="https://colab.research.google.com/github/vaibhavchavhan45/langchain-core-components/blob/main/ContexualCompressorRetriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

In [ ]:
!pip install -U langchain langchain-community langchain-openai langchain-google-genai faiss-cpu chromadb tiktoken groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.6/426.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.9 MB/s eta 0:

In [ ]:
!pip install -U langchain-classic

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [ ]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
embedding = GoogleGenerativeAIEmbeddings(
    model = "models/text-embedding-004"
)

In [ ]:
vector_store = FAISS.from_documents(
    docs,
    embedding
)

In [ ]:
# retriever (an component of main retriever)
base_retriever = vector_store.as_retriever(search_kwargs = {"k" : 3})

In [ ]:
llm = ChatOpenAI(
    model = 'openai/gpt-oss-120b',
)

# compressor (an another component of main retriever)
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
# main retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [ ]:
query = "What is photosynthesis?"

In [ ]:
result = compression_retriever.invoke(query)

In [ ]:
result

[Document(metadata={'source': 'Doc1'}, page_content='Photosynthesis is the process by which green plants convert sunlight into energy.'),
 Document(metadata={'source': 'Doc4'}, page_content='Photosynthesis does not occur in animal cells.'),
 Document(metadata={'source': 'Doc2'}, page_content='The chlorophyll in plant cells captures sunlight during photosynthesis.')]

In [ ]:
for i, item in enumerate(result):
  print(f"\n -- Results {i+1} -- \n")
  print(item.page_content)


 -- Results 1 -- 

Photosynthesis is the process by which green plants convert sunlight into energy.

 -- Results 2 -- 

Photosynthesis does not occur in animal cells.

 -- Results 3 -- 

The chlorophyll in plant cells captures sunlight during photosynthesis.


**Problem statement** : When documents are large and noisy, how do we fetch only the most relevant chunks of information from that document to send to the LLM instead of sending everything(whole document) to the LLM.

Solution : Contextual Compression Retriever

Steps:
1. Documents are stored as embedding vectors in a vector store (FAISS).
2. User query is converted into an embedding vector using the same embedding model.
3. Main retriever (ContextualCompressionRetriever) receives the user query.
4. Main retriever is composed of two components:
    a base retriever
    a compressor
5. Main retriever passes the query to the base retriever.
6. Base retriever performs similarity search on the vector store and retrieves the most relevant documents (not necessarily whole dataset).
7. Base retriever returns those documents to the main retriever.
8. Main retriever sends the retrieved documents along with the query to the compressor (LLMChainExtractor).
9. Compressor feeds (query + documents) to the LLM, which:
    removes irrelevant text
    keeps only the information related to the query
10. Compressor returns the trimmed / relevant chunks to the main retriever.
11. Main retriever returns the final compressed result to the user.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.documents import Document

In [ ]:
docs = [
    Document(page_content=(
        """Machine learning is a subset of artificial intelligence that enables
        computers to learn from data without being explicitly programmed. It uses
        algorithms to identify patterns and make predictions. Common applications
        include image recognition, natural language processing, and recommendation systems."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """Deep learning is a specialized branch of machine learning that uses neural
        networks with multiple layers. These networks can automatically learn hierarchical
        representations of data. Deep learning has revolutionized computer vision, speech
        recognition, and language translation tasks."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Supervised learning is a type of machine learning where the model is trained
        on labeled data. The algorithm learns to map inputs to outputs based on example
        input-output pairs. Common tasks include classification and regression problems."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """Cloud computing provides on-demand access to computing resources over the internet.
        Users can rent servers, storage, and applications instead of maintaining physical
        infrastructure. Major providers include AWS, Azure, and Google Cloud Platform."""
    ), metadata={"source": "Doc4"}),

    Document(page_content=(
        """Blockchain technology is a distributed ledger system that maintains a secure
        and transparent record of transactions. It uses cryptographic techniques to ensure
        data integrity. Applications include cryptocurrency, supply chain management, and
        smart contracts."""
    ), metadata={"source": "Doc5"}),

    Document(page_content=(
        """Reinforcement learning is a machine learning paradigm where agents learn to
        make decisions by interacting with an environment. The agent receives rewards or
        penalties for its actions and aims to maximize cumulative reward over time."""
    ), metadata={"source": "Doc6"}),

    Document(page_content=(
        """Quantum computing leverages quantum mechanical phenomena to perform computations.
        Quantum bits or qubits can exist in superposition states, potentially solving certain
        problems exponentially faster than classical computers. Applications are still being
        explored."""
    ), metadata={"source": "Doc7"}),

    Document(page_content=(
        """Neural networks are computing systems inspired by biological neural networks in
        the brain. They consist of interconnected nodes organized in layers that process
        information. Neural networks are fundamental to modern machine learning and AI."""
    ), metadata={"source": "Doc8"}),

    Document(page_content=(
        """Artificial intelligence refers to the simulation of human intelligence in machines
        that are programmed to think and learn. AI systems can perform tasks such as visual
        perception, speech recognition, decision-making, and language translation. Modern AI
        combines multiple techniques including machine learning, deep learning, and neural networks."""
    ), metadata={"source": "Doc9"}),

    Document(page_content=(
        """Natural language processing enables computers to understand, interpret, and generate
        human language. It combines computational linguistics with machine learning algorithms.
        Applications include chatbots, sentiment analysis, machine translation, and text
        summarization. NLP is a critical component of modern AI systems."""
    ), metadata={"source": "Doc10"}),

    Document(page_content=(
        """Computer vision is a field of artificial intelligence that trains computers to
        interpret and understand visual information from the world. It involves acquiring,
        processing, analyzing, and understanding digital images or videos. Applications include
        facial recognition, autonomous vehicles, and medical image analysis."""
    ), metadata={"source": "Doc11"}),

    Document(page_content=(
        """Cybersecurity involves protecting computer systems, networks, and data from digital
        attacks, unauthorized access, and damage. It includes practices like encryption,
        firewalls, intrusion detection, and security audits. With increasing cyber threats,
        organizations invest heavily in security measures and protocols."""
    ), metadata={"source": "Doc12"}),

    Document(page_content=(
        """Data science combines statistics, mathematics, programming, and domain expertise
        to extract insights from data. Data scientists use techniques like data mining,
        predictive modeling, and visualization. The field has become crucial for business
        intelligence and decision-making across industries."""
    ), metadata={"source": "Doc13"}),

    Document(page_content=(
        """DevOps is a set of practices that combines software development and IT operations
        to shorten the development lifecycle. It emphasizes automation, continuous integration,
        continuous delivery, and collaboration between teams. Popular tools include Docker,
        Kubernetes, Jenkins, and GitLab."""
    ), metadata={"source": "Doc14"}),

    Document(page_content=(
        """Convolutional neural networks are specialized deep learning architectures designed
        for processing grid-like data such as images. CNNs use convolutional layers to
        automatically learn spatial hierarchies of features. They have revolutionized image
        classification, object detection, and facial recognition tasks."""
    ), metadata={"source": "Doc15"}),

    Document(page_content=(
        """Transfer learning is a machine learning technique where a model trained on one task
        is adapted for a second related task. This approach reduces training time and data
        requirements. It's particularly effective in computer vision and natural language
        processing applications where pre-trained models are fine-tuned."""
    ), metadata={"source": "Doc16"}),

    Document(page_content=(
        """Internet of Things refers to the network of physical devices embedded with sensors,
        software, and connectivity to exchange data. IoT applications span smart homes,
        industrial automation, healthcare monitoring, and smart cities. Security and data
        privacy are major concerns in IoT implementations."""
    ), metadata={"source": "Doc17"}),

    Document(page_content=(
        """Edge computing processes data near the source of data generation rather than in
        centralized data centers. This reduces latency, bandwidth usage, and improves response
        times. It's crucial for real-time applications like autonomous vehicles, industrial
        automation, and smart devices."""
    ), metadata={"source": "Doc18"}),

    Document(page_content=(
        """Generative adversarial networks consist of two neural networks competing against
        each other - a generator and a discriminator. GANs can create realistic synthetic data
        including images, videos, and audio. Applications include image generation, style
        transfer, and data augmentation for machine learning."""
    ), metadata={"source": "Doc19"}),

    Document(page_content=(
        """Recurrent neural networks are designed to work with sequential data by maintaining
        internal memory. RNNs process inputs in a sequence and retain information from previous
        steps. They're widely used in time series prediction, speech recognition, and natural
        language processing tasks."""
    ), metadata={"source": "Doc20"})
]

In [ ]:
embedding = GoogleGenerativeAIEmbeddings(
    model = "models/text-embedding-004"
)

In [ ]:
vectorstore = FAISS.from_documents(
    docs,
    embedding
)

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs = {"k" : 10})

In [ ]:
model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
re_ranker = CrossEncoderReranker(model = model, top_n = 5)

In [ ]:
Compressor_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = re_ranker
)

In [ ]:
query = "What is machine learning?"

In [ ]:
response = Compressor_retriever.invoke(query)

In [ ]:
# cheking the content of base_retriever (not part of main flow)
base_retriever_content = base_retriever.invoke(query)

base_retriever_content

[Document(id='57236c57-7137-42af-a03f-33e770091b70', metadata={'source': 'Doc1'}, page_content='Machine learning is a subset of artificial intelligence that enables \n        computers to learn from data without being explicitly programmed. It uses \n        algorithms to identify patterns and make predictions. Common applications \n        include image recognition, natural language processing, and recommendation systems.'),
 Document(id='a69ce9e3-ffca-4313-8d15-5f3e810855a8', metadata={'source': 'Doc2'}, page_content='Deep learning is a specialized branch of machine learning that uses neural \n        networks with multiple layers. These networks can automatically learn hierarchical \n        representations of data. Deep learning has revolutionized computer vision, speech \n        recognition, and language translation tasks.'),
 Document(id='f208a5d1-d735-4667-aa63-5f0c70c54c9a', metadata={'source': 'Doc9'}, page_content='Artificial intelligence refers to the simulation of human in

In [ ]:
# checking the response of main_retriever (not part of main flow)
response

[Document(id='57236c57-7137-42af-a03f-33e770091b70', metadata={'source': 'Doc1'}, page_content='Machine learning is a subset of artificial intelligence that enables \n        computers to learn from data without being explicitly programmed. It uses \n        algorithms to identify patterns and make predictions. Common applications \n        include image recognition, natural language processing, and recommendation systems.'),
 Document(id='ae7c20df-36b3-4a91-9030-5aaa18fdca6a', metadata={'source': 'Doc16'}, page_content="Transfer learning is a machine learning technique where a model trained on one task \n        is adapted for a second related task. This approach reduces training time and data \n        requirements. It's particularly effective in computer vision and natural language \n        processing applications where pre-trained models are fine-tuned."),
 Document(id='19a737fa-ec5e-4dd9-8ba3-dd70feb0e0e0', metadata={'source': 'Doc6'}, page_content='Reinforcement learning is a ma

In [ ]:
for i, item in enumerate(response):
  print(f"\n -- Results {i+1} -- \n")
  print(item.page_content)


 -- Results 1 -- 

Machine learning is a subset of artificial intelligence that enables 
        computers to learn from data without being explicitly programmed. It uses 
        algorithms to identify patterns and make predictions. Common applications 
        include image recognition, natural language processing, and recommendation systems.

 -- Results 2 -- 

Transfer learning is a machine learning technique where a model trained on one task 
        is adapted for a second related task. This approach reduces training time and data 
        requirements. It's particularly effective in computer vision and natural language 
        processing applications where pre-trained models are fine-tuned.

 -- Results 3 -- 

Reinforcement learning is a machine learning paradigm where agents learn to 
        make decisions by interacting with an environment. The agent receives rewards or 
        penalties for its actions and aims to maximize cumulative reward over time.

 -- Results 4 -- 



In the recent above code we are using Re-ranker with Contexual Compression Retriever (instead of compressor)
It has the same flow as of the top code (retriever + compressor)
Main part is re-ranker (Cross Encoder) we are taking from the HF CrossEncoder